# Excel Automation with Python (pandas + openpyxl)

This notebook is designed for a **1-day hands-on training**.

You will learn to:
- Clean messy Excel files automatically  
- Merge multiple Excel files into one  
- Generate a monthly report  
- Export multi-sheet Excel output  
- Apply Excel formatting using `openpyxl`


In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)


## 1) Load a messy Excel file

Sample files:
- `messy_sales_Jan.xlsx`
- `messy_sales_Feb.xlsx`
- `messy_sales_Mar.xlsx`


In [ ]:
file_path = "messy_sales_Jan.xlsx"
raw = pd.read_excel(file_path)

raw.head(10)


## 2) Remove top metadata rows

These files contain **2 non-data rows** at the top (title + note).  
We'll remove them and reset the index.


In [ ]:
df = raw.iloc[2:].copy()
df.reset_index(drop=True, inplace=True)

df.head()


## 3) Clean column names (standardize)

We'll convert to `snake_case`:
- trim spaces
- lowercase
- convert spaces to `_`


In [ ]:
df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace(r"\s+", "_", regex=True)
)

df.columns


## 4) Handle missing values

We will drop rows where key fields are missing: `sales_amount` and `date`.


In [ ]:
df = df.dropna(subset=["sales_amount", "date"]).copy()
df.head()


## 5) Convert data types

- `sales_amount` -> numeric  
- `date` -> datetime


In [ ]:
df["sales_amount"] = pd.to_numeric(df["sales_amount"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df.dtypes


In [ ]:
df


## 6) Merge multiple Excel files from a folder

We will:
- read all `messy_sales_*.xlsx` files  
- clean each file the same way  
- add a `source_file` column  
- concatenate into `master_df`


In [ ]:
folder = Path(".")
files = sorted(folder.glob("messy_sales_*.xlsx"))
files


In [ ]:
def load_and_clean_sales_excel(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path)
    df = raw.iloc[2:].copy()
    df.reset_index(drop=True, inplace=True)
    df.columns = (
        df.columns.astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
    )
    df = df.dropna(subset=["sales_amount", "date"]).copy()
    df["sales_amount"] = pd.to_numeric(df["sales_amount"], errors="coerce")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["source_file"] = path.name
    return df

df_list = [load_and_clean_sales_excel(f) for f in files]
master_df = pd.concat(df_list, ignore_index=True)

master_df.head()


## 7) Generate a monthly report (summary)

Group by month and sum total sales.


In [ ]:
monthly_summary = (
    master_df
      .assign(month=master_df["date"].dt.to_period("M").astype(str))
      .groupby("month", as_index=False)["sales_amount"]
      .sum()
      .rename(columns={"sales_amount": "total_sales"})
)

monthly_summary


## 8) Export to Excel (multi-sheet)

We will export:
- `Clean_Data` (master_df)  
- `Summary` (monthly_summary)


In [ ]:
output_path = "automated_report.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    master_df.to_excel(writer, sheet_name="Clean_Data", index=False)
    monthly_summary.to_excel(writer, sheet_name="Summary", index=False)

output_path


## 9) Apply formatting using openpyxl

We will:
- bold header row  
- freeze top row  
- set auto filter  
- adjust column widths  
- apply number formats (currency + date)


In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

wb = load_workbook(output_path)

def format_sheet(ws, currency_cols=None, date_cols=None):
    currency_cols = currency_cols or []
    date_cols = date_cols or []

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    header_font = Font(bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(vertical="center")

    # Column widths
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        col_letter = get_column_letter(col_idx)
        max_len = 0
        for c in col_cells:
            if c.value is not None:
                max_len = max(max_len, len(str(c.value)))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 40)

    # Number formats based on header names
    headers = [c.value for c in ws[1]]
    for j, header in enumerate(headers, start=1):
        if header in currency_cols:
            for (cell,) in ws.iter_rows(min_row=2, min_col=j, max_col=j):
                cell.number_format = '"RM" #,##0.00'
        if header in date_cols:
            for (cell,) in ws.iter_rows(min_row=2, min_col=j, max_col=j):
                cell.number_format = "yyyy-mm-dd"

format_sheet(wb["Clean_Data"], currency_cols=["sales_amount"], date_cols=["date"])
format_sheet(wb["Summary"], currency_cols=["total_sales"], date_cols=[])

formatted_path = "automated_report_formatted.xlsx"
wb.save(formatted_path)

formatted_path


✅ End-to-end automation complete!
- Clean messy files  
- Merge multiple months  
- Generate monthly report  
- Export to Excel  
- Apply formatting automatically  
